In [1]:
from IPython.display import Audio
import numpy as np
import pandas as pd

signal = np.random.random(750)
Audio(signal, rate=250)

In [19]:
piano_path = "https://raw.githubusercontent.com/tipthederiver/Math-7243-2020/master/Labs/Lab%206/Piano%20Notes.csv"
notes = pd.read_csv(piano_path, encoding= 'unicode_escape')
notes

,Note,Frequency (Hz),Wavelength (cm),Key Position
0,C0,16.35,2109.89,1
1,C#0/Db0,17.32,1991.47,2
2,D0,18.35,1879.69,3
3,D#0/Eb0,19.45,1774.20,4
4,E0,20.60,1674.62,5
...,...,...,...,...
103,G8,6271.93,5.50,104
104,G#8/Ab8,6644.88,5.19,105
105,A8,7040.00,4.90,106
106,A#8/Bb8,7458.62,4.63,107


In [3]:
notes[notes["Key Position"]==1]

,Note,Frequency (Hz),Wavelength (cm),Key Position
0,C0,16.35,2109.89,1


In [17]:
def gen_audio(song, notes, framerate=22050, L=.25):
    N = len(song)
    W = int(framerate*L)
    t = np.linspace(0,L,W)
    data = np.zeros(W*N)

    for i in range(N): 
        F = notes["Frequency (Hz)"].iloc[song[i]+1]
        data[W*i:W*(i+1)] = np.sin(2*np.pi*F*t)
        
    return data

In [5]:
song_0 = [65,65,65,65,72,72,70,70,69,69,67,67,65,65,65,65,72,72,72,72,74,74,74,74,74,74,74]
song_1 = [60,60,60,60,60,60,60,60,60,60,60,60,62,62,64,64,65,65,65,65,65,65,65,65,65,65,65]

In [7]:
data_0 = gen_audio(song_0, notes)
Audio(data_0,rate=22050)

In [8]:
data_0 = gen_audio(song_0, notes)
data_1 = gen_audio(song_1, notes)
Audio(data_0 + data_1,rate=22050)

In [1]:
import pandas as pd
import numpy as np
import os
import glob

base_path = 'jsb_chorales'
train_files = sorted(glob.glob(os.path.join(base_path, 'train', '*.csv')))
valid_files = sorted(glob.glob(os.path.join(base_path, 'valid', '*.csv')))
test_files = sorted(glob.glob(os.path.join(base_path, 'test', '*.csv')))

single_chorale = pd.read_csv(train_files[0])

chorale_array = single_chorale[['note0', 'note1', 'note2', 'note3']].values

In [2]:
def seq(chorale, seq_length=16):
    X = []
    y = []

    for i in range(len(chorale) - seq_length):
        X.append(chorale[i:i + seq_length])
        y.append(chorale[i + 1:i + seq_length + 1])

    return np.array(X), np.array(y)

seq_length = 16

X_single, y_single = seq(chorale_array, seq_length)

print(f"X shape: {X_single.shape}")
print(f"y shape: {y_single.shape}")

X shape: (176, 16, 4)
y shape: (176, 16, 4)


In [3]:
min_note = chorale_array.min()
max_note = chorale_array.max()

X_single_scaled = (X_single - min_note) / (max_note - min_note)
y_single_scaled = (y_single - min_note) / (max_note - min_note)

In [4]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

model_Bach = Sequential()
model_Bach.add(LSTM(64, input_shape=(seq_length, 4), return_sequences=True))
model_Bach.add(Dense(4, activation='sigmoid'))

model_Bach.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'])

model_Bach.summary()

2025-11-09 12:32:27.461511: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2025-11-09 12:32:27.461570: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2025-11-09 12:32:27.461579: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2025-11-09 12:32:27.461622: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-11-09 12:32:27.461638: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/sashwatdesai/Desktop/College/Masters/Northeastern University/SEMESTER WORK/Semester 3/Machine Learning 1 (MATH 7243)/Homework/Lab5/.venv/lib/python3.9/site-packages/keras/src/laye

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 16, 64)         │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16, 4)          │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,924 (70.02 KB)

 Trainable params: 17,924 (70.02 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
history = model_Bach.fit(
    X_single_scaled,
    y_single_scaled,
    epochs=20,
    batch_size=32,
    verbose=1)

Epoch 1/20


2025-11-09 12:32:30.523206: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


6/6 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - loss: 0.0683 - mae: 0.2274
Epoch 2/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0451 - mae: 0.1821 
Epoch 3/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0275 - mae: 0.1359
Epoch 4/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0179 - mae: 0.1036
Epoch 5/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0166 - mae: 0.1014
Epoch 6/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0157 - mae: 0.0988
Epoch 7/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0144 - mae: 0.0938 
Epoch 8/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 0.0136 - mae: 0.0900
Epoch 9/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 0.0128 - mae: 0.0871 
Epoch 10/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0125 - mae: 0.0859 
Epoch 11/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0121 - mae: 0.0842 
Epoch 12/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.0118 - mae: 0.0830 
Epoch 13/20
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 0.0113 - mae: 

In [6]:
def seq_all(file_list, seq_length=16):
    X_full = []
    y_full= []
    
    for file_path in file_list:
        chorale_df = pd.read_csv(file_path)
        chorale_array = chorale_df[['note0', 'note1', 'note2', 'note3']].values
        
        X_temp, y_temp = seq(chorale_array, seq_length)
        if len(X_temp) > 0:
            X_full.extend(X_temp)
            y_full.extend(y_temp)
    
    return np.array(X_full), np.array(y_full)

X_train, y_train = seq_all(train_files, seq_length)
print(f"Training data: {X_train.shape}")

X_valid, y_valid = seq_all(valid_files, seq_length)
print(f"Validation data: {X_valid.shape}")

X_test, y_test = seq_all(test_files, seq_length)
print(f"Test data: {X_test.shape}")

all_notes = np.concatenate([X_train.flatten(), X_valid.flatten(), X_test.flatten()])
min_note = all_notes.min()
max_note = all_notes.max()

X_train_norm = (X_train - min_note) / (max_note - min_note)
y_train_norm = (y_train - min_note) / (max_note - min_note)

X_valid_norm = (X_valid - min_note) / (max_note - min_note)
y_valid_norm = (y_valid - min_note) / (max_note - min_note)

X_test_norm = (X_test - min_note) / (max_note - min_note)
y_test_norm = (y_test - min_note) / (max_note - min_note)

Training data: (51564, 16, 4)
Validation data: (17192, 16, 4)
Test data: (17668, 16, 4)


In [7]:
callbacks = [ModelCheckpoint('model_Bach.keras',
                            monitor='val_loss',
                            save_best_only=True,
                            verbose=1),
            EarlyStopping(monitor='val_loss',
                          patience=10,
                          restore_best_weights=True,
                          verbose=1)]

history_full = model_Bach.fit(
    X_train_norm, y_train_norm,
    validation_data=(X_valid_norm, y_valid_norm),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1)

Epoch 1/50
806/806 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0062 - mae: 0.0502
Epoch 1: val_loss improved from inf to 0.00354, saving model to model_Bach.keras
806/806 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - loss: 0.0062 - mae: 0.0502 - val_loss: 0.0035 - val_mae: 0.0310
Epoch 2/50
799/806 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0017 - mae: 0.0268
Epoch 2: val_loss improved from 0.00354 to 0.00269, saving model to model_Bach.keras
806/806 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0017 - mae: 0.0268 - val_loss: 0.0027 - val_mae: 0.0252
Epoch 3/50
799/806 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0012 - mae: 0.0219
Epoch 3: val_loss improved from 0.00269 to 0.00228, saving model to model_Bach.keras
806/806 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - loss: 0.0012 - mae: 0.0219 - val_loss: 0.0023 - val_mae: 0.0220
Epoch 4/50
800/806 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0010 - mae: 0.0192
Epoch 4: val_loss improved from 0.00228 to 0.00204, saving model to model_Bach.keras
806/806 ━━━━━━━━━━━

In [9]:
test_loss, test_mae = model_Bach.evaluate(X_test_norm, y_test_norm, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f}")

Test Loss: 0.0011
Test MAE: 0.0117


In [24]:
def generate_chorale(model_Bach, X_test_norm):

    seed_idx = np.random.randint(0, len(X_test_norm))
    seed = X_test_norm[seed_idx]
    current_sequence = seed.copy()

    generated = []

    for _ in range(100):
        prediction_sequence = model_Bach.predict(
            current_sequence.reshape(1, seq_length, 4),
            verbose=0
        )

        next_timestep = prediction_sequence[0, -1, :]

        next_timestep = np.clip(next_timestep, 0, 1)

        generated.append(next_timestep)

        current_sequence = np.vstack([current_sequence[1:], next_timestep])

    generated = np.array(generated)  # Shape: (100, 4)

    full_chorale = np.vstack([seed, generated])

    full_chorale = full_chorale * (max_note - min_note) + min_note
    full_chorale = np.round(full_chorale).astype(int)

    pd.DataFrame(full_chorale, columns=['note0', 'note1', 'note2', 'note3']).to_csv('generated_chorale.csv', index=False)

    return full_chorale

chorale = generate_chorale(model_Bach, X_test_norm)

In [20]:
from IPython.display import Audio

generated_song_0 = pd.read_csv("generated_chorale.csv")

col_0 = generated_song_0["note0"]. values
col_1 = generated_song_0["note1"]. values
col_2 = generated_song_0["note2"]. values
col_3 = generated_song_0["note3"]. values

gen_data_0 = gen_audio(col_0, notes)
gen_data_1 = gen_audio(col_1, notes)
gen_data_2 = gen_audio(col_2, notes)
gen_data_3 = gen_audio(col_3, notes)
Audio(gen_data_0 + gen_data_1 + gen_data_2 + gen_data_3,rate=22050)